In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    row_number,
    sha2,
    when,
)
from pyspark.sql.window import Window

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
    sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
    sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()

In [0]:
def load_curated_customerrequest_cepalplatform(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args):
        date_filter = f"businessdate = {run_date!r}"

        cluster_columns = ["BusinessDate"]
        curated_table_path = f"{curated_zone}.Customer_Request"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "cepalplatform",
            )
        else:
            logging.info("Table does not exist, no need to delete entries for business date!")

        # ================== customerrequest (main fact table) ==================

        customerrequest_extra_filters = None

        if customerrequest_extra_filters is not None:
            customerrequest_extra_filters.append(date_filter)
            customerrequest_final_filters = " AND ".join(customerrequest_extra_filters)
        else:
            customerrequest_final_filters = date_filter

        customerrequest_selected_fields = [
            "CustomerRequestID",
            "CustomerRequestCDI",
            "CustomerRequestTypeID",
            "CustomerRequestStatusID",
            "CustomerRequestDateCreated",
            "CustomerRequestDateModified",
            "CustomerRequestDate",
            "CustomerRequestCreatedByUser",
            "CustomerRequesCreatedByUserChannel",
            "CustomerRequestLockedByUserChannel",
            "CustomerRequestLockedByUser",
            "isLocked",
            "CustomerRequestUnlockedByUser",
            "CustomerRequestUnlockedDate",
            "CustomerRequestComplaintID",
            "CustomerRequestInstallmentSuspensionID",
            "CustomerRequestNotesOtherSender",
            "CustomerRequestNotesOtherCC",
            "PhoneNumber",
            "CallTime",
            "CustomerRequestLastModifiedByUser",
            "CustomerRequestSendCode",
            "CustomerRequestEmail",
            "CustomerRequestShippingAddress",
            "CustomerRequestIsComplaint",
            "CustomerRequestFullName",
            "CustomerRequestOracleDate",
            "BusinessDate",
        ]

        customerrequest = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "cepalplatform_customerrequest",
            customerrequest_final_filters,
            customerrequest_selected_fields,
        )

        # ================== customerrequestdata (customer demographics) ==================

        customerrequestdata_extra_filters = None

        if customerrequestdata_extra_filters is not None:
            customerrequestdata_extra_filters.append(date_filter)
            customerrequestdata_final_filters = " AND ".join(customerrequestdata_extra_filters)
        else:
            customerrequestdata_final_filters = date_filter

        customerrequestdata_selected_fields = [
            "CustomerCDI",
            "CustomerAFM",
            "CustomerFirstName",
            "CustomerLastName",
            "CustomerFatherName",
            "IsInLawN3869",
            "PortfolioPrefix",
            "StagingDateInserted",
            "BusinessDate",
        ]

        customerrequestdata_raw = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "cepalplatform_customerrequestdata",
            customerrequestdata_final_filters,
            customerrequestdata_selected_fields,
        )

        # Deduplicate customerrequestdata: keep latest row per CustomerCDI per BusinessDate
        customerrequestdata_window = Window.partitionBy(
            customerrequestdata_raw.CustomerCDI,
            customerrequestdata_raw.BusinessDate,
        ).orderBy(col("StagingDateInserted").desc())

        customerrequestdata = (
            customerrequestdata_raw
            .withColumn("_rn", row_number().over(customerrequestdata_window))
            .filter(col("_rn") == 1)
            .drop("_rn", "StagingDateInserted")
        )

        # ================== rfdtcustomerrequesttype (type lookup) ==================

        requesttype_extra_filters = None

        if requesttype_extra_filters is not None:
            requesttype_extra_filters.append(date_filter)
            requesttype_final_filters = " AND ".join(requesttype_extra_filters)
        else:
            requesttype_final_filters = date_filter

        requesttype_selected_fields = [
            "ID",
            "Category",
            "IsCustomerService",
            "IsLaw3869",
            "StagingDateInserted",
            "BusinessDate",
        ]

        requesttype_raw = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "cepalplatform_rfdtcustomerrequesttype",
            requesttype_final_filters,
            requesttype_selected_fields,
        )

        # Deduplicate requesttype: keep latest row per ID per BusinessDate
        requesttype_window = Window.partitionBy(
            requesttype_raw.ID,
            requesttype_raw.BusinessDate,
        ).orderBy(col("StagingDateInserted").desc())

        requesttype = (
            requesttype_raw
            .withColumn("_rn", row_number().over(requesttype_window))
            .filter(col("_rn") == 1)
            .drop("_rn", "StagingDateInserted")
        )

        # ================== rfdtcustomerrequeststatus (status lookup) ==================

        requeststatus_extra_filters = None

        if requeststatus_extra_filters is not None:
            requeststatus_extra_filters.append(date_filter)
            requeststatus_final_filters = " AND ".join(requeststatus_extra_filters)
        else:
            requeststatus_final_filters = date_filter

        requeststatus_selected_fields = [
            "ID",
            "Status",
            "Priority",
            "IsCustomerService",
            "StagingDateInserted",
            "BusinessDate",
        ]

        requeststatus_raw = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "cepalplatform_rfdtcustomerrequeststatus",
            requeststatus_final_filters,
            requeststatus_selected_fields,
        )

        # Deduplicate requeststatus: keep latest row per ID per BusinessDate
        requeststatus_window = Window.partitionBy(
            requeststatus_raw.ID,
            requeststatus_raw.BusinessDate,
        ).orderBy(col("StagingDateInserted").desc())

        requeststatus = (
            requeststatus_raw
            .withColumn("_rn", row_number().over(requeststatus_window))
            .filter(col("_rn") == 1)
            .drop("_rn", "StagingDateInserted")
        )

        # ========================== Joins ==========================

        customerrequest_customerrequestdata_join_condition = (
            (customerrequest.CustomerRequestCDI == customerrequestdata.CustomerCDI)
            & (customerrequest.BusinessDate == customerrequestdata.BusinessDate)
        )

        customerrequest_requesttype_join_condition = (
            (customerrequest.CustomerRequestTypeID == requesttype.ID)
            & (customerrequest.BusinessDate == requesttype.BusinessDate)
        )

        customerrequest_requeststatus_join_condition = (
            (customerrequest.CustomerRequestStatusID == requeststatus.ID)
            & (customerrequest.BusinessDate == requeststatus.BusinessDate)
        )

        target_table = (
            customerrequest
            .join(
                customerrequestdata,
                customerrequest_customerrequestdata_join_condition,
                "left",
            )
            .join(
                requesttype,
                customerrequest_requesttype_join_condition,
                "left",
            )
            .join(
                requeststatus,
                customerrequest_requeststatus_join_condition,
                "left",
            )
        )

        # ========================== Primary Key ==========================

        primary_key_fields = [
            customerrequest.CustomerRequestID,
            customerrequest.CustomerRequestCDI,
            lit("cepalplatform"),
        ]

        target_table = (
            target_table
            .withColumn(
                "CustomerRequestSurrogateId",
                sha2(concat_ws(";", *primary_key_fields), 256),
            )
            .withColumn(
                "CustomerRequestKey",
                concat_ws(";", customerrequest.CustomerRequestID, customerrequest.CustomerRequestCDI),
            )
        )

        # ========================== Select Final Columns ==========================

        result_df = target_table.select(
            target_table.CustomerRequestSurrogateId.cast("string").alias("CustomerRequestId"),
            target_table.CustomerRequestKey.cast("string").alias("CustomerRequestKey"),
            lit("cepalplatform").cast("string").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias("IngestionTimestamp"),
            lit(None).cast("string").alias("BatchId"),
            customerrequest.CustomerRequestCDI.cast("string").alias("CustomerCDI"),
            customerrequestdata.CustomerAFM.cast("string").alias("CustomerAFM"),
            customerrequest.CustomerRequestFullName.cast("string").alias("CustomerFullName"),
            customerrequestdata.CustomerFirstName.cast("string").alias("CustomerFirstName"),
            customerrequestdata.CustomerLastName.cast("string").alias("CustomerLastName"),
            customerrequestdata.CustomerFatherName.cast("string").alias("CustomerFatherName"),
            customerrequestdata.PortfolioPrefix.cast("string").alias("PortfolioPrefix"),
            customerrequest.CustomerRequestTypeID.cast("int").alias("RequestTypeID"),
            requesttype.Category.cast("string").alias("RequestTypeCategory"),
            requesttype.IsLaw3869.cast("boolean").alias("IsRequestTypeLaw3869"),
            requesttype.IsCustomerService.cast("boolean").alias("IsRequestTypeCustomerService"),
            customerrequest.CustomerRequestStatusID.cast("int").alias("RequestStatusID"),
            requeststatus.Status.cast("string").alias("RequestStatus"),
            requeststatus.Priority.cast("int").alias("RequestStatusPriority"),
            requeststatus.IsCustomerService.cast("boolean").alias("IsStatusCustomerService"),
            lit(None).cast("boolean").alias("IsStatusLaw3869"),
            customerrequest.CustomerRequestIsComplaint.cast("boolean").alias("IsComplaint"),
            customerrequest.isLocked.cast("boolean").alias("IsLocked"),
            customerrequest.CustomerRequestDateCreated.cast("timestamp").alias("CreatedDate"),
            customerrequest.CustomerRequestDateModified.cast("timestamp").alias("ModifiedDate"),
            customerrequest.CustomerRequestDate.cast("string").alias("RequestDate"),
            customerrequest.CustomerRequestCreatedByUser.cast("string").alias("CreatedByUser"),
            customerrequest.CustomerRequesCreatedByUserChannel.cast("string").alias("CreatedByUserChannel"),
            customerrequest.CustomerRequestLastModifiedByUser.cast("string").alias("LastModifiedByUser"),
            customerrequest.CustomerRequestLockedByUser.cast("string").alias("LockedByUser"),
            customerrequest.CustomerRequestLockedByUserChannel.cast("string").alias("LockedByUserChannel"),
            customerrequest.CustomerRequestUnlockedByUser.cast("string").alias("UnlockedByUser"),
            customerrequest.CustomerRequestUnlockedDate.cast("timestamp").alias("UnlockedDate"),
            customerrequest.PhoneNumber.cast("string").alias("PhoneNumber"),
            customerrequest.CustomerRequestEmail.cast("string").alias("Email"),
            customerrequest.CustomerRequestShippingAddress.cast("string").alias("ShippingAddress"),
            customerrequest.CustomerRequestComplaintID.cast("int").alias("ComplaintID"),
            customerrequest.CustomerRequestInstallmentSuspensionID.cast("int").alias("InstallmentSuspensionID"),
            customerrequest.CustomerRequestOracleDate.cast("timestamp").alias("OracleDate"),
            customerrequest.CallTime.cast("timestamp").alias("CallTime"),
            customerrequest.CustomerRequestSendCode.cast("bigint").alias("SendCode"),
            customerrequest.CustomerRequestNotesOtherSender.cast("string").alias("NotesOtherSender"),
            customerrequest.CustomerRequestNotesOtherCC.cast("string").alias("NotesOtherCC"),
            customerrequestdata.IsInLawN3869.cast("boolean").alias("IsInLawN3869"),
            customerrequest.BusinessDate.alias("BusinessDate"),
        )

        # ========================== Append =========================

        result_df.write.format("delta").clusterBy("BusinessDate").mode("append").saveAsTable(
            curated_table_path
        )

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")

In [0]:
load_curated_customerrequest_cepalplatform(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)